# Steph Curry Shot Probabilities (2022–2023)

This notebook reproduces the Udacity/Woolf Nanodegree probability calculations using the Steph Curry shot dataset.


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import binom

CSV_PATH = '../data/steph_curry_shots_2023.csv'
df = pd.read_csv(CSV_PATH)
df.head()


In [ ]:
# Normalize booleans (handles TRUE/FALSE as strings)
def to_bool(s):
    if pd.api.types.is_bool_dtype(s):
        return s
    return s.astype(str).str.strip().str.upper().map({'TRUE': True, 'FALSE': False, '1': True, '0': False})

df['result'] = to_bool(df['result'])
df['lead'] = to_bool(df['lead'])

n = len(df)
n


## Overall shooting statistics


In [ ]:
p_make = (df['result'] == True).sum() / n
p_miss = 1 - p_make

p3 = (df['shot_type'] == 3).sum() / n
p2 = 1 - p3

p_make, p_miss, p3, p2


## Binomial probabilities
- P(exactly 3 makes in next 4 shots)
- P(exactly 4 threes in next 5 shots)


In [ ]:
p_3_of_4_makes = binom.pmf(k=3, n=4, p=p_make)
p_4_of_5_threes = binom.pmf(k=4, n=5, p=p3)
p_3_of_4_makes, p_4_of_5_threes


## Conditional probabilities (Future) using Bayes-style ratios


In [ ]:
# Joint probabilities
p_make_and_3 = ((df['result'] == True) & (df['shot_type'] == 3)).sum() / n
p_make_and_2 = ((df['result'] == True) & (df['shot_type'] == 2)).sum() / n
p_lead_and_3 = ((df['lead'] == True) & (df['shot_type'] == 3)).sum() / n
p_lead_and_2 = ((df['lead'] == True) & (df['shot_type'] == 2)).sum() / n

# Conditionals
p_make_given_3 = p_make_and_3 / p3
p_make_given_2 = p_make_and_2 / p2
p_lead_given_3 = p_lead_and_3 / p3
p_lead_given_2 = p_lead_and_2 / p2

p_make_given_3, p_lead_given_3, p_make_given_2, p_lead_given_2


## Conditional probabilities (Retrospective) — Bayes
Given Steph just **made** a shot: compute P(3|Made) and P(2|Made).


In [ ]:
p3_given_make = p_make_and_3 / p_make
p2_given_make = 1 - p3_given_make
p3_given_make, p2_given_make, p3_given_make + p2_given_make


## Summary table


In [ ]:
summary = {
    'P(Make)': p_make,
    'P(Miss)': p_miss,
    'P(3)': p3,
    'P(2)': p2,
    'P(X=3 makes in 4 shots)': p_3_of_4_makes,
    'P(X=4 threes in 5 shots)': p_4_of_5_threes,
    'P(Make|3)': p_make_given_3,
    'P(Lead|3)': p_lead_given_3,
    'P(Make|2)': p_make_given_2,
    'P(Lead|2)': p_lead_given_2,
    'P(3|Made)': p3_given_make,
    'P(2|Made)': p2_given_make,
}
pd.Series(summary).to_frame('value').round(4)
